In [105]:
import pyspark
import pyspark.sql.functions as F

In [106]:
spark = pyspark.sql.SparkSession.builder.master("local[4]").appName("spark_data").config(
            "spark.executor.memory", "15g").config("spark.driver.memory", "15g").getOrCreate()

In [107]:
df = spark.read.parquet("./silver/labeled_training_set_no_tensor.parquet")

In [108]:
df.show()

+--------------------+--------------------+----------------+--------------------+----------------+--------------------+----------+------+---------+-----+----------+-----------+---------+----------+--------+----------+--------------+-------------+----------+
|                name|           imageData|prelabeled_fraud|                  id|     person_name|             address|  birthday|gender|ethnicity|class|issue_date|expire_date|eye_color|hair_color|is_donor|is_veteran|license_number|height_inches|weight_lbs|
+--------------------+--------------------+----------------+--------------------+----------------+--------------------+----------+------+---------+-----+----------+-----------+---------+----------+--------+----------+--------------+-------------+----------+
|generated.photos_...|iVBORw0KGgoAAAANS...|               0|generated.photos_...|  JASMINE BRYANT|588 Camelback Roa...|1987-04-30|     F|    black|    A|2022-01-18| 2027-01-18|      BRO|       BRO|   FALSE|     FALSE|     D118

In [109]:
df.select('license_number').count()

998

In [110]:
df.select('license_number').distinct().count()

884

In [111]:
spark.sql('select count(*) from {df} where license_number is null', df=df).show()

+--------+
|count(1)|
+--------+
|       0|
+--------+



In [112]:
lic_df = df.select('license_number')

In [113]:
new_doc_number = 5

alt_docs = lic_df


for x in range(new_doc_number-1):
    alt_docs = alt_docs.unionAll(lic_df)

In [114]:
alt_docs.show()

+--------------+
|license_number|
+--------------+
|     D11847007|
|     D83539241|
|     D59828589|
|     D67866458|
|     D02322299|
|     D43874830|
|     D12066765|
|     D71993980|
|     D47497422|
|     D34017127|
|     D85087477|
|     D33339124|
|     D24696139|
|     D33812900|
|     D01295746|
|     D68244025|
|     D33354178|
|     D13036789|
|     D82321386|
|     D93864196|
+--------------+
only showing top 20 rows


In [115]:
alt_docs.select('license_number').count()

4990

In [116]:
alt_docs.select('license_number').distinct().count()

884

In [117]:
#labeled_training_set.write.mode("overwrite").parquet("./silver/labeled_training_set_no_tensor_10k.parquet")

In [118]:
from pyspark.sql.types import StructField, StructType, StringType, IntegerType, DecimalType

In [119]:
schema = StructType([
    StructField("license_number", StringType(), True),
    StructField("plate_number", StringType(), True),
    StructField("photo_id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("toll_id", StringType(), True),
    StructField("speed", IntegerType(), True),
    StructField("pass_device_type", StringType(), True),
    StructField("driver_hair_color", StringType(), True),
    StructField("driver_eye_color", StringType(), True),
    StructField("driver_weight", StringType(), True),
    StructField("toll_amount_usd", DecimalType(8,2), True),
])

In [120]:
from faker import Faker

In [121]:
fake = Faker()

In [122]:
from datetime import datetime

@F.udf(StringType())
def generate_plate_num():
    return fake.license_plate()

@F.udf(StringType())
def generate_photo_id():
    return fake.uuid4()

@F.udf(StringType())
def generate_timestamp ():
    fake_date = fake.date_time_between_dates(datetime(2024,1,1), datetime(2024,12,31))
    return fake_date.strftime("%Y-%m-%d %H:%M:%S")

@F.udf(StringType())
def generate_toll_id():
    return fake.pyint(1,100000)


@F.udf(StringType())
def generate_speed():
    return fake.pyint(25,100)

@F.udf(StringType())
def generate_pass_device_type():
    return fake.pyint(1,4)

@F.udf(StringType())
def generate_driver_hair_color():
    hair_colors = ['BRO','BLK', 'BLN']
    return fake.random_element(hair_colors)

@F.udf(StringType())
def generate_driver_eye_color():
    eye_colors = ['BRO','BLU', 'GRY']
    return fake.random_element(eye_colors)

@F.udf(StringType())
def generate_driver_weight():
    return fake.pyint(70,300)


@F.udf(StringType())
def generate_toll_amount_usd():
    amount = fake.pydecimal(min_value=5,max_value=90, right_digits=2)
    return amount



In [123]:
alt_docs = alt_docs.withColumn('plate_number', generate_plate_num())
alt_docs = alt_docs.withColumn('toll_id', generate_toll_id())
alt_docs = alt_docs.withColumn('timestamp', generate_timestamp())
alt_docs = alt_docs.withColumn('photo_id', generate_photo_id())
alt_docs = alt_docs.withColumn('speed', generate_speed())
alt_docs = alt_docs.withColumn('pass_device_type', generate_pass_device_type())
alt_docs = alt_docs.withColumn('driver_hair_color', generate_driver_hair_color())
alt_docs = alt_docs.withColumn('driver_eye_color', generate_driver_eye_color())
alt_docs = alt_docs.withColumn('driver_weight', generate_driver_weight())
alt_docs = alt_docs.withColumn('toll_amount_usd', generate_toll_amount_usd())


In [124]:
alt_docs.show()

+--------------+------------+-------+-------------------+--------------------+-----+----------------+-----------------+----------------+-------------+---------------+
|license_number|plate_number|toll_id|          timestamp|            photo_id|speed|pass_device_type|driver_hair_color|driver_eye_color|driver_weight|toll_amount_usd|
+--------------+------------+-------+-------------------+--------------------+-----+----------------+-----------------+----------------+-------------+---------------+
|     D11847007|    PPN 4564|  69457|2024-07-12 10:01:36|6a425ba5-7a92-4cf...|   92|               4|              BLN|             GRY|          205|          72.77|
|     D83539241|     0-U7808|  62571|2024-06-23 18:12:34|62a742b7-5abc-46d...|   86|               4|              BLK|             BLU|          192|          58.45|
|     D59828589|     N04-63W|  62757|2024-04-18 21:38:10|a5c075f4-6e60-453...|   86|               4|              BLK|             BLU|          192|          54.46

In [125]:
print(fake.license_plate())

PPN 4564


In [126]:
for x in range(5):
    print(fake.license_plate())

0-U7808
N04-63W
7677 ZQ
295-LPK
777 6AD


In [127]:
alt_docs.write.mode("overwrite").parquet("./silver/alt_docs_x5.parquet")